## Input and output formats

earthkit-plots is designed to be **format-agnostic**: the same plotting code works regardless of where your data comes from or what format it is stored in. This notebook covers the most common input formats and shows how to save your plots to various output formats.

### Input formats

earthkit-plots accepts data from four main sources:

| Source | How to load |
|--------|-------------|
| GRIB | `ekd.from_source("file", "data.grib")` |
| NetCDF | `ekd.from_source("file", "data.nc")` |
| xarray | `ekd.from_source(...).to_xarray()` or any `xr.DataArray` / `xr.Dataset` |
| NumPy | Plain `numpy` arrays with explicit `x=`, `y=` coordinates |

Whichever format you use, **the plotting call stays the same** — only the data loading line changes.

In [ ]:
import earthkit.data as ekd

import earthkit.plots as ekp

### GRIB

GRIB (GRIdded Binary) is the standard format for numerical weather prediction output at operational centres such as ECMWF. GRIB files carry rich metadata — parameter name, units, level type, valid time — which earthkit-plots uses automatically for titles, unit conversion and style selection.

Load a GRIB file with `ekd.from_source`. For a local file, use `ekd.from_source("file", "/path/to/data.grib")`.

In [ ]:
grib = ekd.from_source("sample", "era5-monthly-mean-2t-199312.grib")
grib

In [ ]:
chart = ekp.Map(domain="Europe")
chart.contourf(grib, units="celsius", style="auto")
chart.coastlines()
chart.legend(label="{variable_name} ({units})")
chart.title("{variable_name} – {time:%B %Y}")
chart.show()

### NetCDF

NetCDF is one of the most widely used formats in climate science and follows the CF conventions. earthkit-plots reads netCDF files via earthkit-data and extracts coordinates and metadata automatically.

In [ ]:
nc = ekd.from_source("sample", "era5-monthly-mean-2t-199312.nc")
nc

In [ ]:
chart = ekp.Map(domain="Europe")
chart.contourf(nc, units="celsius", style="auto")
chart.coastlines()
chart.legend(label="{variable_name} ({units})")
chart.title("{variable_name} – {time:%B %Y}")
chart.show()

### xarray

xarray `DataArray` and `Dataset` objects are accepted directly — no conversion step is required. This is the most common path when you want to pre-process data (slicing, arithmetic, resampling) before plotting.

You can convert from an earthkit-data object using `.to_xarray()`, or pass any xarray object you have created yourself.

In [ ]:
ds = ekd.from_source("sample", "era5-monthly-mean-2t-199312.nc").to_xarray()
ds

Pass the `Dataset` directly, or select a single `DataArray` by variable name:

In [ ]:
chart = ekp.Map(domain="Europe")
chart.contourf(ds["t2m"], units="celsius", style="auto")
chart.coastlines()
chart.legend(label="{variable_name} ({units})")
chart.title("{variable_name} – {time:%B %Y}")
chart.show()

### NumPy arrays

Plain NumPy arrays are also supported. The trade-off is that NumPy arrays carry no geographical or meteorological metadata, so you must supply coordinate arrays explicitly via `x=` and `y=`. Unit conversion and automatic titles require an optional `metadata` dict.

The simplest way to get coordinates for a GRIB or netCDF field is from earthkit-data:

In [ ]:
from datetime import datetime

fl = ekd.from_source("sample", "era5-monthly-mean-2t-199312.grib").to_fieldlist()
lats, lons = fl.geography.latlons()
t2m = fl.to_numpy().squeeze()

print(f"data shape: {t2m.shape}, lats: {lats.shape}, lons: {lons.shape}")

Without metadata the plot still works, but titles and unit conversion are unavailable:

In [ ]:
chart = ekp.Map(domain="Europe")
chart.contourf(t2m, x=lons, y=lats)
chart.coastlines()
chart.legend()
chart.show()

Supplying a `metadata` dict restores automatic titles, unit conversion and auto-styles. Valid keys mirror CF-convention attributes — `units`, `long_name`, `time`, and so on:

In [ ]:
metadata = {
    "units": "K",
    "long_name": "2 metre temperature",
    "time": datetime(1993, 12, 1),
}

chart = ekp.Map(domain="Europe")
chart.contourf(t2m, x=lons, y=lats, metadata=metadata, units="celsius", style="auto")
chart.coastlines()
chart.legend(label="{variable_name} ({units})")
chart.title("{variable_name} – {time:%B %Y}")
chart.show()

### Format agnosticism

The four examples above produce identical plots. The only difference is the data loading line — the plotting code is unchanged:

```python
# GRIB
data = ekd.from_source("sample", "era5-monthly-mean-2t-199312.grib")

# NetCDF
data = ekd.from_source("sample", "era5-monthly-mean-2t-199312.nc")

# xarray
data = ekd.from_source("sample", "era5-monthly-mean-2t-199312.nc").to_xarray()["t2m"]

# NumPy (extra setup required)
fl = ekd.from_source("sample", "era5-monthly-mean-2t-199312.grib").to_fieldlist()
lats, lons = fl.geography.latlons()
data = fl.to_numpy().squeeze()

# In every case, the plot call is the same:
chart = ekp.Map(domain="Europe")
chart.contourf(data, units="celsius", style="auto")  # add x=, y=, metadata= for NumPy
```

This format agnosticism is a deliberate design goal of earthkit-plots: your visualisation code should not need to change just because your data arrives in a different format.

---

### Output formats

earthkit-plots wraps matplotlib's [savefig](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html), so you can save to any format that matplotlib supports. The output format is inferred from the file extension.

Use `figure.save()` in place of `figure.show()`:

In [ ]:
chart = ekp.Map(domain="Europe")
chart.contourf(grib, units="celsius", style="auto")
chart.coastlines()
chart.legend(label="{variable_name} ({units})")
chart.title("{variable_name} – {time:%B %Y}")

# PNG — raster, good for web and presentations
chart.save("temperature.png", dpi=150)

# PDF — vector, good for publications and print
chart.save("temperature.pdf")

# SVG — vector, good for further editing in Inkscape / Illustrator
chart.save("temperature.svg")

Key arguments:

- **`dpi`** — dots per inch for raster formats (PNG, JPEG). 150 is good for screen; 300 for print.
- **`bbox_inches`** — defaults to `"tight"`, which crops whitespace around the figure. Pass `None` to use the figure's exact size.
- Any other keyword argument accepted by [`matplotlib.pyplot.savefig`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html) is passed through.

Supported formats include: `png`, `pdf`, `svg`, `eps`, `jpeg`, `tiff`, and more — see the [matplotlib documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.savefig.html) for the full list.